In [25]:
import psutil
import os

def check_open_files():
    """Проверяет открытые файлы текущим процессом"""
    
    print("🔍 ПРОВЕРКА ОТКРЫТЫХ ФАЙЛОВ")
    print("="*50)
    
    current_pid = os.getpid()
    process = psutil.Process(current_pid)
    
    try:
        open_files = process.open_files()
        
        if open_files:
            print(f"Открытых файлов: {len(open_files)}")
            for f in open_files[:10]:  # первые 10
                print(f"  📄 {os.path.basename(f.path)}")
        else:
            print("✅ Нет открытых файлов")
            
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        print("⚠️  Не удалось проверить открытые файлы")

# Установите psutil если нет
# !pip install psutil

In [23]:
import pandas as pd
import os
import glob

folder_path = r'\\vra.local\Root\Public\ОИС\Системы отчетности и анализа данных\Производительность'
target_columns = ['Отобрано паллетов', 'Расчёт отобрано упаковок', 'Отобрано ШТ']

print("=" * 70)
print("✅ ОБРАБОТКА ВСЕХ ЛИСТОВ И СОХРАНЕНИЕ В SQL_READY (.xlsx формат)")
print("=" * 70)

# Находим файлы
all_files = glob.glob(os.path.join(folder_path, 'ВыгрузкаТрудозатрат*.xlsx'))
print(f"Найдено .xlsx файлов: {len(all_files)}")

# Создаем папку SQL_READY
output_dir = os.path.join(folder_path, "SQL_READY")
os.makedirs(output_dir, exist_ok=True)
print(f"Папка для результатов: {output_dir}\n")

processed = 0
skipped = 0

for file_path in all_files:
    filename = os.path.basename(file_path)
    output_filename = filename.replace('.xlsx', '_converted.xlsx')
    output_path = os.path.join(output_dir, output_filename)

    print(f"📁 Обработка: {filename}")

    try:
        # 1. ЧИТАЕМ ВСЕ ЛИСТЫ .XLSX ФАЙЛА
        xl = pd.ExcelFile(file_path, engine='openpyxl')
        sheets_dict = pd.read_excel(xl, sheet_name=None)
        print(f"   📊 Найдено листов: {len(sheets_dict)}")

        # 2. ОБРАБАТЫВАЕМ КАЖДЫЙ ЛИСТ
        with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            for sheet_name, df in sheets_dict.items():
                # 2.1. ПЕРВЫЙ СТОЛБЕЦ = F1 (если в листе есть столбцы)
                if len(df.columns) > 0:
                    original_name = df.columns[0]
                    df = df.rename(columns={original_name: 'F1'})
                
                # 2.2. ПРЕОБРАЗУЕМ ЦЕЛЕВЫЕ КОЛОНКИ В INT64
                for col in target_columns:
                    if col in df.columns:
                        # Безопасное преобразование: нечисловые значения -> NaN -> 0
                        df[col] = pd.to_numeric(df[col], errors='coerce')
                        df[col] = df[col].fillna(0).astype('int64')
                
                # 2.3. ЗАПИСЫВАЕМ ОБРАБОТАННЫЙ ЛИСТ В НОВЫЙ ФАЙЛ
                df.to_excel(writer, sheet_name=sheet_name, index=False)
                print(f"      ✓ Лист '{sheet_name}' обработан")

        # Проверяем результат
        file_size = os.path.getsize(output_path)
        print(f"   ✅ Файл сохранён: {output_filename} ({file_size:,} байт)\n")
        processed += 1

    except Exception as e:
        print(f"   ❌ Ошибка при обработке: {str(e)[:100]}")
        skipped += 1

print("=" * 70)
print(f"📊 ИТОГИ:")
print(f"   Успешно обработано: {processed}")
print(f"   Пропущено: {skipped}")
print(f"   Результаты в папке: {output_dir}")

if processed > 0:
    print(f"\n{'=' * 70}")
    print(f"📋 ДАЛЬНЕЙШИЕ ДЕЙСТВИЯ:")
    print(f"1. Откройте папку: {output_dir}")
    print(f"2. Скопируйте ПУТЬ к этой папке")
    print(f"3. Откройте Excel и запустите макрос ниже")

✅ ОБРАБОТКА ВСЕХ ЛИСТОВ И СОХРАНЕНИЕ В SQL_READY (.xlsx формат)
Найдено .xlsx файлов: 74
Папка для результатов: \\vra.local\Root\Public\ОИС\Системы отчетности и анализа данных\Производительность\SQL_READY

📁 Обработка: ВыгрузкаТрудозатрат01_02_11.xlsx
   📊 Найдено листов: 4
      ✓ Лист 'Лист' обработан
      ✓ Лист 'Лист_2' обработан
      ✓ Лист 'Лист_3' обработан
      ✓ Лист 'Лист_4' обработан
   ✅ Файл сохранён: ВыгрузкаТрудозатрат01_02_11_converted.xlsx (30,164 байт)

📁 Обработка: ВыгрузкаТрудозатрат01_02_12.xlsx
   📊 Найдено листов: 4
      ✓ Лист 'Лист' обработан
      ✓ Лист 'Лист_2' обработан
      ✓ Лист 'Лист_3' обработан
      ✓ Лист 'Лист_4' обработан
   ✅ Файл сохранён: ВыгрузкаТрудозатрат01_02_12_converted.xlsx (36,414 байт)

📁 Обработка: ВыгрузкаТрудозатрат02_03_11.xlsx
   📊 Найдено листов: 4
      ✓ Лист 'Лист' обработан
      ✓ Лист 'Лист_2' обработан
      ✓ Лист 'Лист_3' обработан
      ✓ Лист 'Лист_4' обработан
   ✅ Файл сохранён: ВыгрузкаТрудозатрат02_03_11_conve

In [9]:
def check_first_columns():
    """Проверяет первый столбец в обработанных файлах"""
    
    print("\n" + "="*70)
    print("🔍 ПРОВЕРКА ПЕРВЫХ СТОЛБЦОВ")
    print("="*70)
    
    processed_files = glob.glob(os.path.join(output_dir, '*.xlsx'))
    
    for file_path in processed_files[:3]:  # Проверяем первые 3 файла
        filename = os.path.basename(file_path)
        print(f"\n📊 {filename}:")
        
        try:
            df = pd.read_excel(file_path)
            
            if len(df.columns) > 0:
                first_col = df.columns[0]
                print(f"  Первый столбец: '{first_col}'")
                
                # Показываем несколько значений первого столбца
                if len(df) > 0:
                    sample_values = df[first_col].head(3).tolist()
                    print(f"  Примеры значений: {sample_values}")
            else:
                print("  ⚠️  В файле нет столбцов")
                
        except Exception as e:
            print(f"  ❌ Ошибка: {str(e)[:50]}")

# Запустите проверку после обработки
check_first_columns()


🔍 ПРОВЕРКА ПЕРВЫХ СТОЛБЦОВ

📊 ВыгрузкаТрудозатрат01_02_11.xlsx:
  Первый столбец: 'Исполнитель'
  Примеры значений: ['Склад', 'Исполнитель', 'Заказ']

📊 ВыгрузкаТрудозатрат01_02_12.xlsx:
  Первый столбец: 'Исполнитель'
  Примеры значений: ['Склад', 'Исполнитель', 'Заказ']

📊 ВыгрузкаТрудозатрат02_03_11.xlsx:
  Первый столбец: 'Исполнитель'
  Примеры значений: ['Склад', 'Исполнитель', 'Заказ']
